In [1]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_contents, fetch_website_links
from openai import OpenAI

In [2]:
load_dotenv(override=True)
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL,api_key="ollama")
MODEL = "llama3.2"

In [3]:
links = fetch_website_links("https://detik.com")
links

['#',
 'https://www.detik.com/?tagfrom=framebar',
 'https://www.detik.com/terpopuler',
 'https://20.detik.com/live/',
 'https://news.detik.com/kolom/kirim/',
 'https://news.detik.com/',
 'https://finance.detik.com/',
 'https://inet.detik.com/',
 'https://hot.detik.com/',
 'https://sport.detik.com/',
 'https://sport.detik.com/sepakbola',
 'https://oto.detik.com/',
 'https://travel.detik.com/',
 'https://food.detik.com/',
 'https://health.detik.com/',
 'https://wolipop.detik.com',
 'https://news.detik.com/x/',
 'https://20.detik.com',
 'https://foto.detik.com',
 'https://www.detik.com/edu/',
 'https://www.detik.com/hikmah/',
 'https://www.detik.com/properti/',
 'https://www.detik.com/pop/',
 'https://www.detik.com/jateng',
 'https://www.detik.com/jatim',
 'https://www.detik.com/jabar',
 'https://www.detik.com/sulsel',
 'https://www.detik.com/sumut',
 'https://www.detik.com/bali',
 'https://www.detik.com/sumbagsel',
 'https://www.detik.com/jogja',
 'https://www.detik.com/kalimantan/',
 'h

#### Use a call to LLM to read the links on a webpage, and respond in structured JSON.  

It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://detik.com"))


Here is the list of links on the website https://detik.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#
https://www.detik.com/?tagfrom=framebar
https://www.detik.com/terpopuler
https://20.detik.com/live/
https://news.detik.com/kolom/kirim/
https://news.detik.com/
https://finance.detik.com/
https://inet.detik.com/
https://hot.detik.com/
https://sport.detik.com/
https://sport.detik.com/sepakbola
https://oto.detik.com/
https://travel.detik.com/
https://food.detik.com/
https://health.detik.com/
https://wolipop.detik.com
https://news.detik.com/x/
https://20.detik.com
https://foto.detik.com
https://www.detik.com/edu/
https://www.detik.com/hikmah/
https://www.detik.com/properti/
https://www.detik.com/pop/
https://www.detik.com/jateng
https://www.detik.com/jatim
https://www.detik.com/jabar
https://www.detik

In [7]:
def select_relevant_links(url):
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [10]:
select_relevant_links("https://detik.com")

{'dateCreated': None,
 'description': 'untuk mengakses konten di Detik.com, Anda dapat mengunjungi alamat https://www.detik.com',
 'url': 'https://www.detik.com/',
 'interests': ['Berita',
  'Laravel',
  'Kesehatan',
  'Komedi',
  'Ekonomi (lifestyle)',
  'Fotografi',
  'Membuat Contoh',
  'Bisnis',
  'Istilah',
  'Teknik Sunting',
  'Sipil Inggris Inggris',
  'Perangkap Berita',
  'Pertempuran Berita',
  'Rantauan Berita',
  'Rumahan Inggris',
  'Trik Faktur',
  'Wibudin',
  'Faktur',
  'Penyelidikan Penyelidikan',
  'Istirahat (Laravel)',
  'Komentar',
  'Blogger',
  'Kerja Sama',
  'Bahasa Inggris',
  'Teknik Untuk',
  'Majalah Detik.',
  'Sumber Informasi',
  'Detik Hot',
  'Detik Fashion',
  'Detik Kesehatan',
  'Detik Bisnis',
  'Detik Bahari',
  'Detik Masyarakat',
  'Aplikasi yang Bisa Anda Gunakan',
  'Aplikasi untuk Berita',
  'Kota terbaik di Indonesia',
  'Istilah Istilah',
  'Laravel',
  'Pertajangan Istirahat Kegiatan Penulisan Jurnal Puisi Kontemporer',
  'Mengenail Isti

In [16]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2
Found 9 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/'},
  {'type': 'company page', 'url': 'https://docs.huggingface.co/'},
  {'type': 'blog', 'url': 'https://blog.huggingface.co/'},
  {'type': 'forum/discussion', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'github', 'url': 'https://github.com/huggingface'},
  {'type': 'twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'blog', 'url': 'https://status.huggingface.co/'},
  {'type': 'site', 'url': 'https://huggingface.co/'}]}

In [11]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

### Next step: make the brochure!
Assemble all the details into another prompt to LLM

In [17]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [20]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling llama3.2
Found 9 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
bytedance-research/Lance
Updated
about 6 hours ago
•
1.23k
•
671
Supertone/supertonic-3
Updated
5 days ago
•
40.4k
•
601
tencent/Hy-MT2-1.8B
Updated
1 day ago
•
2.56k
•
329
openbmb/MiniCPM-V-4.6
Updated
4 days ago
•
247k
•
910
tencent/Hy-MT2-30B-A3B
Updated
1 day ago
•
970
•
282
Browse 2M+ models

In [21]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [22]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [23]:
#get_brochure_user_prompt("HuggingFace", "https://huggingface.co")
get_brochure_user_prompt("Detik", "https://detik.com")

Selecting relevant links for https://detik.com by calling llama3.2


KeyError: 'links'

In [24]:
def create_brochure(company_name, url):
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role":"system","content": brochure_system_prompt},
            {"role":"user","content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [26]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2
Found 7 relevant links


# Welcome to Hugging Face

**Building a Community that Drives AI Innovation**

At Hugging Face, we are dedicated to creating a platform where the machine learning community can collaborate, share, and drive innovation in AI. Our mission is to empower developers, researchers, and businesses to unlock the full potential of artificial intelligence.

**What We Do**

* **Build Models**: Host and deploy unlimited public models, datasets, and applications on our collaboration platform.
* **Explore AI Apps**: Browse and try out thousands of pre-trained models for various tasks such as natural language processing (NLP), computer vision, and more.
* **Work with Datasets**: Store and manage large datasets for your projects, and easily retrieve or update them with our dataset management tools.

**Our Core Values**

At Hugging Face, we are guided by the following values:

* **Innovation**: We believe that creativity, collaboration, and constant learning are essential to drive AI innovation.
* **Accessibility**: We aim to make it easy for everyone to access powerful algorithms and tools, regardless of their background or expertise.
* **Transparency**: We prioritize open-source contributions, transparency in our data collection policies, and thorough documentation.

**Customer Success Stories**

Some notable customers who have successfully leveraged Hugging Face include:

* Tesla, which uses Hugging Face's transformers for natural language processing tasks.
* Netflix, which relies on Hugging Face's text classification models to improve content recommendation algorithms.
* The US Department of Defense, which utilizes our models to automate data labeling and enrich national datasets.

**Join Our Community**

Whether you are a researcher, entrepreneur, or student looking to apply AI concepts in your work or projects, we invite you to join the Hugging Face community. We offer various resources for getting started with building AI applications on Hugging Face:

* **Official Documentation**: Visit our documentation portal to explore the API endpoints and guide us through best practices.
* **Guides & Tutorials**: Dive into comprehensive tutorials to master various AI models, algorithms, and techniques.
* **GitHub**: Explore open-source contributions from community members who are passionate about building the future of AI together.

**Join Our Team**

If you share our passion for driving AI innovation and contributing to making machine learning accessible to everyone, consider joining our team. We offer diverse opportunities across fields such as:

* Research
- Development
- Support & Technical Services

Discover a career where collaboration meets technology development, creativity flows through constant innovation, and a community-driven platform empowers individuals to drive the future of AI.



We are excited for you to be part of this exciting adventure!

## Minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation using STREAM and DELTA

In [33]:
def stream_brochure(company_name, url):
    stream = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2
Found 11 relevant links


ConnectionError: HTTPSConnectionPool(host='blog.huggingface.co', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000002896970C290>: Failed to resolve 'blog.huggingface.co' ([Errno 11001] getaddrinfo failed)"))

: 